# 📊 Pandas groupby 완전정복
## 경상북도 안동시 상하수도 요금정보 데이터 분석

### 수업 목표
- `groupby()`의 기본 개념과 동작 원리를 이해한다
- 다양한 집계 함수(`sum`, `mean`, `count`, `min`, `max` 등)를 활용할 수 있다
- `agg()`, `transform()`, `filter()`, `apply()`의 차이를 이해하고 활용할 수 있다
- 다중 컬럼 그룹화와 멀티인덱스를 다룰 수 있다
- 고급 groupby 패턴 (조건부 집계, 누적합, 이상치 탐지 등)을 활용할 수 있다

### 수업 구성 (3시간)
| 시간 | 내용 |
|------|------|
| 1교시 (60분) | 데이터 탐색, groupby 기초, 단일 집계 함수 |
| 2교시 (60분) | agg(), 다중 컬럼 그룹화, transform(), filter() |
| 3교시 (60분) | apply(), 고급 groupby 기법, 실전 연습문제 |

---

# 🕐 1교시: 데이터 탐색과 groupby 기초
---
## 1-1. 라이브러리 임포트 및 데이터 로드

In [1]:
# 필요한 라이브러리 임포트
import pandas as pd
import numpy as np

# pandas 출력 옵션 설정
pd.set_option('display.max_columns', 10)    # 최대 표시 컬럼 수
pd.set_option('display.max_rows', 30)       # 최대 표시 행 수
pd.set_option('display.float_format', '{:,.1f}'.format)  # 숫자 천단위 콤마

print("라이브러리 로드 완료!")

라이브러리 로드 완료!


In [2]:
# CSV 파일 로드 (cp949 인코딩: 한글 윈도우에서 생성된 파일)
df = pd.read_csv('pandas/data/경상북도 안동시_상하수도요금정보_20251124.csv', encoding='cp949')

# 데이터 크기 확인
print(f"데이터 크기: {df.shape[0]:,}행 × {df.shape[1]}열")
df.head(10)

데이터 크기: 471,984행 × 6열


,수용가번호,읍면동명,업종,사용량,사용요금,납기마감일
0,1001020000,중구동,일반용,14,13310,2025-11-24
1,1001030000,중구동,일반용,2,1900,2025-11-24
2,1001040000,중구동,일반용,152,315890,2025-11-24
3,1001050000,중구동,일반용,2,1900,2025-11-24
4,1001060000,중구동,일반용,31,38060,2025-11-24
5,1001065000,중구동,일반용,51,73140,2025-11-24
6,1001070000,중구동,일반겸용,35,25270,2025-11-24
7,1001080000,중구동,일반용,5,4750,2025-11-24
8,1001090000,중구동,일반겸용,26,15180,2025-11-24
9,1001100000,중구동,일반용,109,204220,2025-11-24


## 1-2. 데이터 탐색 (EDA)
> 분석에 앞서 데이터의 구조, 타입, 결측치 등을 확인합니다.

In [3]:
# 데이터 기본 정보 확인
# - 컬럼별 데이터 타입, 결측치 여부 확인
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 471984 entries, 0 to 471983
Data columns (total 6 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   수용가번호   471984 non-null  int64 
 1   읍면동명    471984 non-null  object
 2   업종      471984 non-null  object
 3   사용량     471984 non-null  int64 
 4   사용요금    471984 non-null  int64 
 5   납기마감일   471984 non-null  object
dtypes: int64(3), object(3)
memory usage: 21.6+ MB


In [4]:
# 수치형 컬럼의 기술통계량 확인
# - count, mean, std, min, 25%, 50%, 75%, max 값 확인
df.describe()

,수용가번호,사용량,사용요금
count,"471,984.0","471,984.0","471,984.0"
mean,"21,520,120,883.5",37.5,"38,691.2"
std,"15,783,324,156.6",468.8,"524,407.0"
min,"1,001,020,000.0",0.0,0.0
25%,"5,100,143,750.2",2.0,"1,420.0"
50%,"31,210,710,050.0",10.0,"4,740.0"
75%,"36,882,412,500.0",20.0,"14,670.0"
max,"42,130,460,000.0","56,970.0","59,818,500.0"


In [5]:
df.describe(include='all')

,수용가번호,읍면동명,업종,사용량,사용요금,납기마감일
count,"471,984.0",471984,471984,"471,984.0","471,984.0",471984
unique,NaN,22,6,NaN,NaN,1
top,NaN,풍산읍,가정용,NaN,NaN,2025-11-24
freq,NaN,41112,372864,NaN,NaN,471984
mean,"21,520,120,883.5",NaN,NaN,37.5,"38,691.2",NaN
std,"15,783,324,156.6",NaN,NaN,468.8,"524,407.0",NaN
min,"1,001,020,000.0",NaN,NaN,0.0,0.0,NaN
25%,"5,100,143,750.2",NaN,NaN,2.0,"1,420.0",NaN
50%,"31,210,710,050.0",NaN,NaN,10.0,"4,740.0",NaN
75%,"36,882,412,500.0",NaN,NaN,20.0,"14,670.0",NaN


In [6]:
# 범주형 컬럼의 고유값 확인
# - 읍면동명: 안동시 내 행정구역
# - 업종: 수도 사용 용도 구분
print("📍 읍면동명 고유값:", df['읍면동명'].nunique(), "개")
print(df['읍면동명'].unique())
print()
print("🏭 업종 고유값:", df['업종'].nunique(), "개")
print(df['업종'].unique())

📍 읍면동명 고유값: 22 개
['중구동' '명륜동' '용상동' '서구동' '태화동' '강남동' '평화동' '안기동' '옥동' '송현동' '풍산읍' '일직면'
 '임동면' '도산면' '길안면' '서후면' '풍천면' '와룡면' '남후면' '북후면' '녹전면' '예안면']

🏭 업종 고유값: 6 개
['일반용' '일반겸용' '가정용' '산업용' '욕탕용' '욕탕겸용']


In [7]:
# 각 범주형 컬럼의 값 분포 확인
# - value_counts()로 각 값이 몇 번 등장하는지 확인
print("=== 읍면동명별 데이터 건수 ===")
print(df['읍면동명'].value_counts())
print()
print("=== 업종별 데이터 건수 ===")
print(df['업종'].value_counts())

=== 읍면동명별 데이터 건수 ===
읍면동명
풍산읍    41112
길안면    40056
태화동    38892
용상동    38604
서구동    32700
풍천면    32064
와룡면    26004
중구동    25248
서후면    21528
일직면    20940
북후면    20160
강남동    17112
평화동    16212
송현동    15348
남후면    13896
안기동    13152
명륜동    12792
녹전면    11244
임동면    10944
옥동     10656
도산면     7836
예안면     5484
Name: count, dtype: int64

=== 업종별 데이터 건수 ===
업종
가정용     372864
일반용      85104
일반겸용     11892
산업용       1920
욕탕용        192
욕탕겸용        12
Name: count, dtype: int64


## 1-3. groupby() 기본 개념

### groupby란?
- **Split-Apply-Combine** 패턴의 구현
  1. **Split**: 특정 기준(컬럼)으로 데이터를 그룹으로 나눔
  2. **Apply**: 각 그룹에 함수(집계, 변환 등)를 적용
  3. **Combine**: 결과를 하나로 합침

```
전체 데이터 → [그룹1, 그룹2, ...] → [결과1, 결과2, ...] → 최종 결과
```

### 기본 문법
```python
df.groupby('그룹기준컬럼')['집계대상컬럼'].집계함수()
```

In [8]:
# groupby 객체 생성 - 아직 연산이 수행되지 않은 상태 (지연 평가)
# groupby()만 호출하면 DataFrameGroupBy 객체가 반환됨
grouped = df.groupby('업종')
print(type(grouped))
print(f"그룹 수: {grouped.ngroups}")
print(f"그룹 키: {list(grouped.groups.keys())}")

<class 'pandas.core.groupby.generic.DataFrameGroupBy'>
그룹 수: 6
그룹 키: ['가정용', '산업용', '욕탕겸용', '욕탕용', '일반겸용', '일반용']


In [9]:
# 그룹별 데이터 확인 - 각 그룹에 어떤 데이터가 포함되어 있는지 순회
# for문으로 각 그룹의 이름과 데이터를 확인할 수 있음
for name, group in grouped:
    print(f"=== 그룹: {name} (데이터 수: {len(group)}건) ===")
    print(group.head(3))  # 각 그룹의 상위 3건만 출력
    print()

=== 그룹: 가정용 (데이터 수: 372864건) ===
         수용가번호 읍면동명   업종  사용량   사용요금       납기마감일
12  1001130000  중구동  가정용   61  73970  2025-11-24
21  1001220000  중구동  가정용    0      0  2025-11-24
24  1001250000  중구동  가정용    7   3310  2025-11-24

=== 그룹: 산업용 (데이터 수: 1920건) ===
           수용가번호 읍면동명   업종  사용량    사용요금       납기마감일
265   1005090000  중구동  산업용    1    1050  2025-11-24
4325  3150122500  용상동  산업용  181  190050  2025-11-24
4715  3190940001  용상동  산업용   42   44100  2025-11-24

=== 그룹: 욕탕겸용 (데이터 수: 12건) ===
            수용가번호 읍면동명    업종  사용량    사용요금       납기마감일
7689   4190010001  서구동  욕탕겸용  185  159440  2025-11-24
47021  4190010001  서구동  욕탕겸용  221  192520  2025-11-24
86353  4190010001  서구동  욕탕겸용  177  152080  2025-11-24

=== 그룹: 욕탕용 (데이터 수: 192건) ===
           수용가번호 읍면동명   업종  사용량   사용요금       납기마감일
471   1008310000  중구동  욕탕용    0      0  2025-11-24
864   1014410000  중구동  욕탕용   20  18380  2025-11-24
2220  2020001000  명륜동  욕탕용   38  34920  2025-11-24

=== 그룹: 일반겸용 (데이터 수: 11892건) ===
         수용가번호 

In [11]:
# 특정 그룹만 선택하여 확인
# get_group()으로 특정 그룹의 데이터프레임을 추출할 수 있음
산업용_df = grouped.get_group('산업용')
print(f"산업용 데이터 수: {len(산업용_df)}건")
산업용_df.head()

산업용 데이터 수: 1920건


,수용가번호,읍면동명,업종,사용량,사용요금,납기마감일
265,1005090000,중구동,산업용,1,1050,2025-11-24
4325,3150122500,용상동,산업용,181,190050,2025-11-24
4715,3190940001,용상동,산업용,42,44100,2025-11-24
5775,3430050001,용상동,산업용,81,85050,2025-11-24
5802,3430230001,용상동,산업용,49,51450,2025-11-24


In [12]:
indust_df = df[df['업종']=='산업용']
print(f"산업용 데이터 수: {len(indust_df)}건")
indust_df.head()

산업용 데이터 수: 1920건


,수용가번호,읍면동명,업종,사용량,사용요금,납기마감일
265,1005090000,중구동,산업용,1,1050,2025-11-24
4325,3150122500,용상동,산업용,181,190050,2025-11-24
4715,3190940001,용상동,산업용,42,44100,2025-11-24
5775,3430050001,용상동,산업용,81,85050,2025-11-24
5802,3430230001,용상동,산업용,49,51450,2025-11-24


## 1-4. 단일 집계 함수 활용

> groupby 이후 사용할 수 있는 주요 집계 함수:
> 
> | 함수 | 설명 |
> |------|------|
> | `sum()` | 합계 |
> | `mean()` | 평균 |
> | `median()` | 중앙값 |
> | `count()` | 개수 |
> | `size()` | 그룹 크기 (NaN 포함) |
> | `min()` | 최솟값 |
> | `max()` | 최댓값 |
> | `std()` | 표준편차 |
> | `var()` | 분산 |
> | `first()` / `last()` | 첫 번째 / 마지막 값 |

In [13]:
# sum() - 업종별 총 사용량 합계
# 어떤 업종이 가장 많은 물을 사용하는지 확인
df.groupby('업종')['사용량'].sum()

업종
가정용     11268881
산업용      1911018
욕탕겸용        2111
욕탕용        39433
일반겸용      749954
일반용      3726925
Name: 사용량, dtype: int64

In [14]:
# mean() - 업종별 평균 사용요금
# 업종에 따라 평균 요금이 얼마나 차이나는지 확인
df.groupby('업종')['사용요금'].mean().round(0)

업종
가정용       20,931.0
산업용    1,005,597.0
욕탕겸용     140,338.0
욕탕용      342,253.0
일반겸용      70,927.0
일반용       89,487.0
Name: 사용요금, dtype: float64

In [15]:
# count() vs size() 비교
# count()는 NaN을 제외한 개수, size()는 NaN 포함 전체 개수
print("=== count() - NaN 제외 ===")
print(df.groupby('업종')['사용량'].count())
print()
print("=== size() - NaN 포함 ===")
print(df.groupby('업종').size())

=== count() - NaN 제외 ===
업종
가정용     372864
산업용       1920
욕탕겸용        12
욕탕용        192
일반겸용     11892
일반용      85104
Name: 사용량, dtype: int64

=== size() - NaN 포함 ===
업종
가정용     372864
산업용       1920
욕탕겸용        12
욕탕용        192
일반겸용     11892
일반용      85104
dtype: int64


In [16]:
# max(), min() - 읍면동별 최대/최소 사용요금
# 각 지역에서 가장 높은 요금과 가장 낮은 요금 확인
print("=== 읍면동별 최대 사용요금 (상위 10개) ===")
print(df.groupby('읍면동명')['사용요금'].max().sort_values(ascending=False).head(10))
print()
print("=== 읍면동별 최소 사용요금 ===")
print(df.groupby('읍면동명')['사용요금'].min())

=== 읍면동별 최대 사용요금 (상위 10개) ===
읍면동명
풍산읍    59818500
용상동    38508690
강남동    34907880
일직면    34113590
풍천면    27299370
송현동    20945010
서구동    18940650
임동면    14203260
옥동     13747340
중구동     8208420
Name: 사용요금, dtype: int64

=== 읍면동별 최소 사용요금 ===
읍면동명
강남동    0
길안면    0
남후면    0
녹전면    0
도산면    0
명륜동    0
북후면    0
서구동    0
서후면    0
송현동    0
안기동    0
예안면    0
옥동     0
와룡면    0
용상동    0
일직면    0
임동면    0
중구동    0
태화동    0
평화동    0
풍산읍    0
풍천면    0
Name: 사용요금, dtype: int64


In [17]:
# median() - 읍면동별 사용량 중앙값
# 평균은 이상치에 민감하므로, 중앙값으로 대표값을 비교
df.groupby('읍면동명')['사용량'].median().sort_values(ascending=False)

읍면동명
옥동    46.0
용상동   16.0
강남동   14.0
평화동   14.0
태화동   13.0
송현동   13.0
안기동   13.0
중구동   11.0
서구동   10.0
길안면    8.0
풍산읍    8.0
풍천면    8.0
명륜동    8.0
서후면    7.0
북후면    7.0
일직면    7.0
도산면    7.0
예안면    7.0
와룡면    6.0
임동면    6.0
녹전면    6.0
남후면    6.0
Name: 사용량, dtype: float64

In [18]:
# std() - 업종별 사용요금의 표준편차
# 표준편차가 크면 그룹 내 편차가 크다는 의미
df.groupby('업종')['사용요금'].std().round(0)

업종
가정용      263,583.0
산업용    4,928,737.0
욕탕겸용      40,970.0
욕탕용    1,079,381.0
일반겸용   1,000,213.0
일반용      710,769.0
Name: 사용요금, dtype: float64

In [19]:
# 여러 컬럼에 동시에 집계 적용
# 대상 컬럼을 리스트로 지정하면 여러 컬럼의 집계를 한 번에 수행
df.groupby('업종')[['사용량', '사용요금']].mean().round(0)

,사용량,사용요금
업종,,
가정용,30.0,"20,931.0"
산업용,995.0,"1,005,597.0"
욕탕겸용,176.0,"140,338.0"
욕탕용,205.0,"342,253.0"
일반겸용,63.0,"70,927.0"
일반용,44.0,"89,487.0"


In [20]:
# describe() - 그룹별 기술통계량 한 번에 보기
# 각 그룹의 count, mean, std, min, 25%, 50%, 75%, max를 한눈에 확인
df.groupby('업종')['사용요금'].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
업종,,,,,,,,
가정용,"372,864.0","20,931.0","263,583.0",0.0,940.0,"4,260.0","11,360.0","35,969,730.0"
산업용,"1,920.0","1,005,597.0","4,928,737.0",0.0,"5,250.0","27,300.0","134,400.0","59,818,500.0"
욕탕겸용,12.0,"140,338.0","40,970.0","27,670.0","132,782.0","151,165.0","158,060.0","192,520.0"
욕탕용,192.0,"342,253.0","1,079,381.0",0.0,0.0,"18,835.0","65,470.0","6,628,310.0"
일반겸용,"11,892.0","70,927.0","1,000,213.0",0.0,"9,480.0","20,850.0","41,040.0","42,896,660.0"
일반용,"85,104.0","89,487.0","710,769.0",0.0,"2,850.0","13,310.0","41,520.0","38,508,690.0"


# 🕐 2교시: agg(), 다중 그룹화, transform(), filter()
---
## 2-1. agg() - 여러 집계 함수 동시 적용

> `agg()`는 하나의 groupby에 여러 집계 함수를 동시에 적용할 수 있는 강력한 메서드입니다.

### 사용 방법
```python
# 방법 1: 리스트로 여러 함수 적용
df.groupby('기준')['대상'].agg(['sum', 'mean', 'count'])

# 방법 2: 딕셔너리로 컬럼별 다른 함수 적용
df.groupby('기준').agg({'컬럼A': 'sum', '컬럼B': 'mean'})

# 방법 3: Named Aggregation (컬럼명 지정)
df.groupby('기준').agg(새이름=('컬럼', '함수'))
```

In [21]:
# 방법 1: 리스트로 여러 집계 함수를 한 컬럼에 동시 적용
# 사용요금에 대해 합계, 평균, 최대, 최소를 한 번에 계산
df.groupby('업종')['사용요금'].agg(['sum', 'mean', 'max', 'min', 'count'])

,sum,mean,max,min,count
업종,,,,,
가정용,7804348570,"20,930.8",35969730,0,372864
산업용,1930745950,"1,005,596.8",59818500,0,1920
욕탕겸용,1684060,"140,338.3",192520,27670,12
욕탕용,65712610,"342,253.2",6628310,0,192
일반겸용,843467300,"70,927.3",42896660,0,11892
일반용,7615664760,"89,486.6",38508690,0,85104


In [22]:
# 방법 2: 딕셔너리로 컬럼마다 서로 다른 집계 함수 적용
# 사용량은 합계와 평균, 사용요금은 합계와 최대값
df.groupby('업종').agg({
    '사용량': ['sum', 'mean'],     # 사용량: 합계와 평균
    '사용요금': ['sum', 'max']     # 사용요금: 합계와 최대값
}).round(0)

사용량              사용요금          
           sum  mean         sum       max
업종                                        
가정용   11268881  30.0  7804348570  35969730
산업용    1911018 995.0  1930745950  59818500
욕탕겸용      2111 176.0     1684060    192520
욕탕용      39433 205.0    65712610   6628310
일반겸용    749954  63.0   843467300  42896660
일반용    3726925  44.0  7615664760  38508690

In [23]:
# 방법 3: Named Aggregation - 결과 컬럼명을 직접 지정 (가독성 향상)
# pd.NamedAgg 또는 튜플 형태로 (컬럼, 함수)를 지정
df.groupby('업종').agg(
    총사용량=('사용량', 'sum'),           # 결과 컬럼명을 '총사용량'으로 지정
    평균사용량=('사용량', 'mean'),         # 결과 컬럼명을 '평균사용량'으로 지정
    총요금=('사용요금', 'sum'),            # 결과 컬럼명을 '총요금'으로 지정
    평균요금=('사용요금', 'mean'),         # 결과 컬럼명을 '평균요금'으로 지정
    건수=('사용량', 'count')              # 결과 컬럼명을 '건수'로 지정
).round(0)

,총사용량,평균사용량,총요금,평균요금,건수
업종,,,,,
가정용,11268881,30.0,7804348570,"20,931.0",372864
산업용,1911018,995.0,1930745950,"1,005,597.0",1920
욕탕겸용,2111,176.0,1684060,"140,338.0",12
욕탕용,39433,205.0,65712610,"342,253.0",192
일반겸용,749954,63.0,843467300,"70,927.0",11892
일반용,3726925,44.0,7615664760,"89,487.0",85104


In [24]:
# 사용자 정의 함수(lambda)를 agg에 적용
# 예: 각 그룹의 사용요금 범위(최대-최소)를 구하는 커스텀 함수
df.groupby('업종')['사용요금'].agg(
    요금범위=lambda x: x.max() - x.min(),   # 최대값 - 최소값
    상위10퍼센트=lambda x: x.quantile(0.9)   # 90번째 백분위수
).round(0)

,요금범위,상위10퍼센트
업종,,
가정용,35969730,"23,500.0"
산업용,59818500,"737,940.0"
욕탕겸용,164850,"166,055.0"
욕탕용,6628310,"628,960.0"
일반겸용,42896660,"77,478.0"
일반용,38508690,"121,500.0"


## 2-2. 다중 컬럼으로 그룹화

> 두 개 이상의 컬럼을 기준으로 그룹화하면 더 세밀한 분석이 가능합니다.
> 
> ```python
> df.groupby(['컬럼1', '컬럼2'])['대상'].집계함수()
> ```
> 결과는 **멀티인덱스(MultiIndex)** 형태로 반환됩니다.

In [25]:
# 읍면동명 + 업종으로 다중 그룹화
# 각 지역의 업종별 평균 사용요금 확인
result = df.groupby(['읍면동명', '업종'])['사용요금'].mean().round(0)
print(type(result.index))  # 멀티인덱스 확인
result.head(15)

<class 'pandas.core.indexes.multi.MultiIndex'>


읍면동명  업종  
강남동   가정용     58,755.0
      산업용    108,672.0
      일반겸용    38,440.0
      일반용    109,056.0
길안면   가정용     10,110.0
      산업용    348,973.0
      욕탕용    363,344.0
      일반겸용   123,631.0
      일반용     66,731.0
남후면   가정용      8,438.0
      산업용    101,003.0
      일반용     90,550.0
녹전면   가정용      6,686.0
      산업용     44,250.0
      일반용     19,649.0
Name: 사용요금, dtype: float64

In [26]:
# unstack() - 멀티인덱스를 피벗하여 보기 좋은 표 형태로 변환
# 행: 읍면동명, 열: 업종 으로 교차표 생성
result_table = df.groupby(['읍면동명', '업종'])['사용요금'].mean().round(0).unstack(fill_value=0)
result_table

업종,가정용,산업용,욕탕겸용,욕탕용,일반겸용,일반용
읍면동명,,,,,,
강남동,"58,755.0","108,672.0",0.0,0.0,"38,440.0","109,056.0"
길안면,"10,110.0","348,973.0",0.0,"363,344.0","123,631.0","66,731.0"
남후면,"8,438.0","101,003.0",0.0,0.0,0.0,"90,550.0"
녹전면,"6,686.0","44,250.0",0.0,0.0,0.0,"19,649.0"
도산면,"7,579.0",0.0,0.0,0.0,"18,393.0","118,086.0"
명륜동,"22,315.0",0.0,0.0,"1,415,821.0","35,663.0","64,260.0"
북후면,"8,652.0","1,069,134.0",0.0,0.0,0.0,"43,329.0"
서구동,"20,956.0","8,713.0","140,338.0","594,531.0","25,595.0","71,590.0"
서후면,"9,192.0","84,344.0",0.0,0.0,"24,865.0","54,131.0"


In [27]:
# reset_index() - 멀티인덱스를 일반 컬럼으로 변환
# 분석 결과를 다시 일반 DataFrame으로 사용하고 싶을 때 유용
df.groupby(['읍면동명', '업종']).agg(
    평균사용량=('사용량', 'mean'),
    평균요금=('사용요금', 'mean'),
    건수=('수용가번호', 'count')
).round(0).reset_index().head(10)

,읍면동명,업종,평균사용량,평균요금,건수
0,강남동,가정용,84.0,"58,755.0",9084
1,강남동,산업용,113.0,"108,672.0",276
2,강남동,일반겸용,51.0,"38,440.0",1056
3,강남동,일반용,51.0,"109,056.0",6696
4,길안면,가정용,12.0,"10,110.0",36420
5,길안면,산업용,362.0,"348,973.0",204
6,길안면,욕탕용,334.0,"363,344.0",12
7,길안면,일반겸용,36.0,"123,631.0",60
8,길안면,일반용,36.0,"66,731.0",3360
9,남후면,가정용,10.0,"8,438.0",12648


In [28]:
# as_index=False 옵션 - reset_index() 없이 바로 일반 DataFrame으로 반환
# groupby의 기준 컬럼이 인덱스가 아닌 일반 컬럼으로 유지됨
df.groupby('업종', as_index=False)[['사용량', '사용요금']].mean().round(0)

,업종,사용량,사용요금
0,가정용,30.0,"20,931.0"
1,산업용,995.0,"1,005,597.0"
2,욕탕겸용,176.0,"140,338.0"
3,욕탕용,205.0,"342,253.0"
4,일반겸용,63.0,"70,927.0"
5,일반용,44.0,"89,487.0"


## 2-3. transform() - 원본 크기 유지 변환

> `transform()`은 그룹별 연산 결과를 **원본 DataFrame과 같은 크기**로 반환합니다.
> 
> - 집계(`agg`)는 그룹 수만큼 줄어든 결과를 반환
> - 변환(`transform`)은 원본과 동일한 행 수를 유지
> 
> **활용 사례**: 그룹 평균과의 차이 계산, 그룹 내 정규화, 그룹별 비율 계산 등

In [29]:
# transform 기본 예제 - 각 행에 해당 업종의 평균 사용요금을 추가
# 원본 DataFrame의 행 수(471,984)와 동일한 크기의 결과 반환
df['업종별_평균요금'] = df.groupby('업종')['사용요금'].transform('mean')

# 원본 데이터와 그룹 평균이 같은 행에 나란히 표시됨
df[['읍면동명', '업종', '사용요금', '업종별_평균요금']].head(10)

,읍면동명,업종,사용요금,업종별_평균요금
0,중구동,일반용,13310,"89,486.6"
1,중구동,일반용,1900,"89,486.6"
2,중구동,일반용,315890,"89,486.6"
3,중구동,일반용,1900,"89,486.6"
4,중구동,일반용,38060,"89,486.6"
5,중구동,일반용,73140,"89,486.6"
6,중구동,일반겸용,25270,"70,927.3"
7,중구동,일반용,4750,"89,486.6"
8,중구동,일반겸용,15180,"70,927.3"
9,중구동,일반용,204220,"89,486.6"


In [30]:
# transform 활용 1 - 그룹 평균과의 차이(편차) 계산
# 각 수용가의 요금이 해당 업종 평균 대비 얼마나 높은지/낮은지 확인
df['평균대비_차이'] = df['사용요금'] - df.groupby('업종')['사용요금'].transform('mean')

# 평균보다 요금이 높은 수용가 확인
print("=== 업종 평균 대비 요금이 높은 상위 10건 ===")
df.nlargest(10, '평균대비_차이')[['읍면동명', '업종', '사용요금', '업종별_평균요금', '평균대비_차이']]

=== 업종 평균 대비 요금이 높은 상위 10건 ===


,읍면동명,업종,사용요금,업종별_평균요금,평균대비_차이
414013,풍산읍,산업용,59818500,"1,005,596.8","58,812,903.2"
374681,풍산읍,산업용,57435000,"1,005,596.8","56,429,403.2"
60022,풍산읍,산업용,53088000,"1,005,596.8","52,082,403.2"
453345,풍산읍,산업용,51735600,"1,005,596.8","50,730,003.2"
178018,풍산읍,산업용,49191450,"1,005,596.8","48,185,853.2"
20690,풍산읍,산업용,48702150,"1,005,596.8","47,696,553.2"
217351,풍산읍,산업용,46019400,"1,005,596.8","45,013,803.2"
256682,풍산읍,산업용,45222450,"1,005,596.8","44,216,853.2"
99354,풍산읍,산업용,44131500,"1,005,596.8","43,125,903.2"
451283,풍산읍,일반겸용,42896660,"70,927.3","42,825,732.7"


In [31]:
# transform 활용 2 - 그룹 내 비율 계산
# 각 수용가의 사용량이 해당 읍면동 전체 사용량에서 차지하는 비율
df['동내_사용량비율'] = df['사용량'] / df.groupby('읍면동명')['사용량'].transform('sum') * 100

# 각 읍면동에서 사용량 비율이 높은 수용가 (대량 사용자)
print("=== 동내 사용량 비율 상위 10건 ===")
df.nlargest(10, '동내_사용량비율')[['읍면동명', '업종', '사용량', '동내_사용량비율']].round(2)

=== 동내 사용량 비율 상위 10건 ===


,읍면동명,업종,사용량,동내_사용량비율
102036,일직면,일반용,12289,4.4
377667,임동면,일반용,5145,2.9
416999,임동면,일반용,5021,2.8
259671,임동면,일반용,4750,2.7
220339,임동면,일반용,4713,2.6
102343,임동면,일반용,4627,2.6
299003,임동면,일반용,4435,2.5
141675,임동면,일반용,4272,2.4
414013,풍산읍,산업용,56970,2.4
456331,임동면,일반용,4236,2.4


In [ ]:
# transform 활용 3 - 그룹 내 순위 매기기
# rank()를 transform과 함께 사용하여 각 업종 내에서의 요금 순위
df['업종내_요금순위'] = df.groupby('업종')['사용요금'].rank(ascending=False, method='min')

# 각 업종에서 요금이 가장 높은 수용가 (순위 1위)
print("=== 업종별 요금 1위 수용가 ===")
df[df['업종내_요금순위'] == 1][['읍면동명', '업종', '사용량', '사용요금', '업종내_요금순위']]

## 2-4. filter() - 조건에 맞는 그룹만 선택

> `filter()`는 그룹 단위로 조건을 평가하여, **조건을 만족하는 그룹의 전체 데이터**를 반환합니다.
> 
> - 일반 불리언 인덱싱: 행 단위 필터링
> - `groupby().filter()`: 그룹 단위 필터링
> 
> ```python
> df.groupby('기준').filter(lambda x: 조건식)
> ```

In [ ]:
# filter 예제 1 - 수용가 수가 30,000건 이상인 읍면동만 추출
# 데이터가 충분히 많은 지역만 분석하고 싶을 때 유용
df_large = df.groupby('읍면동명').filter(lambda x: len(x) >= 30000)

print(f"원본 데이터: {len(df):,}건")
print(f"필터 후 데이터: {len(df_large):,}건")
print(f"포함된 읍면동: {df_large['읍면동명'].unique()}")

In [ ]:
# filter 예제 2 - 평균 사용요금이 50,000원 이상인 업종만 추출
# 고액 사용 업종만 따로 분석하고 싶을 때
df_high_fee = df.groupby('업종').filter(lambda x: x['사용요금'].mean() >= 50000)

print(f"원본 데이터: {len(df):,}건")
print(f"필터 후 데이터: {len(df_high_fee):,}건")
print(f"포함된 업종: {df_high_fee['업종'].unique()}")
print()
# 확인: 해당 업종들의 평균 요금
df_high_fee.groupby('업종')['사용요금'].mean().round(0)

In [ ]:
# filter 예제 3 - 사용량 표준편차가 100 이상인 읍면동만 추출
# 사용 패턴의 편차가 큰 지역만 관심이 있을 때
df_high_std = df.groupby('읍면동명').filter(lambda x: x['사용량'].std() >= 100)

print(f"필터 후 데이터: {len(df_high_std):,}건")
print(f"포함된 읍면동: {df_high_std['읍면동명'].unique()}")

# 🕐 3교시: apply(), 고급 groupby 기법, 실전 연습문제
---
## 3-1. apply() - 그룹별 사용자 정의 함수 적용

> `apply()`는 각 그룹에 임의의 함수를 적용할 수 있는 가장 유연한 메서드입니다.
> 
> | 메서드 | 반환 크기 | 용도 |
> |--------|----------|------|
> | `agg()` | 그룹 수 | 집계 (요약) |
> | `transform()` | 원본과 동일 | 변환 (브로드캐스트) |
> | `filter()` | 원본 이하 | 그룹 필터링 |
> | `apply()` | 자유 | 무엇이든 가능 |

In [ ]:
# apply 예제 1 - 각 업종별 사용요금 상위 3건 추출
# nlargest()를 사용하여 그룹별 Top N을 구하는 패턴
def top3_by_fee(group):
    """각 그룹에서 사용요금 상위 3건을 반환하는 함수"""
    return group.nlargest(3, '사용요금')

# 업종별 사용요금 상위 3건
df.groupby('업종').apply(top3_by_fee, include_groups=False)[['읍면동명', '사용량', '사용요금']]

In [ ]:
# apply 예제 2 - 그룹별 요약 정보를 Series로 반환
# 각 그룹에 대해 여러 통계를 한 번에 계산하여 반환
def group_summary(group):
    """각 그룹에 대한 요약 통계를 Series로 반환하는 함수"""
    return pd.Series({
        '총건수': len(group),
        '총사용량': group['사용량'].sum(),
        '평균사용량': group['사용량'].mean(),
        '총요금': group['사용요금'].sum(),
        '평균요금': group['사용요금'].mean(),
        '요금중앙값': group['사용요금'].median(),
        '최대사용자_요금': group['사용요금'].max(),
        '무사용_건수': (group['사용량'] == 0).sum()  # 사용량이 0인 건수
    })

# 읍면동별 요약 통계
df.groupby('읍면동명').apply(group_summary, include_groups=False).round(0)

In [ ]:
# apply 예제 3 - 그룹 내 정규화 (Min-Max Scaling)
# 각 읍면동 그룹 내에서 사용요금을 0~1 범위로 정규화
def min_max_normalize(group):
    """그룹 내 사용요금을 0~1 범위로 정규화하는 함수"""
    min_val = group['사용요금'].min()
    max_val = group['사용요금'].max()
    # 최대/최소가 같으면 0으로 처리 (분모가 0이 되는 것 방지)
    if max_val == min_val:
        group['정규화_요금'] = 0
    else:
        group['정규화_요금'] = (group['사용요금'] - min_val) / (max_val - min_val)
    return group

# 읍면동별 정규화 수행
df_normalized = df.groupby('읍면동명').apply(min_max_normalize, include_groups=False)
df_normalized[['읍면동명', '업종', '사용요금', '정규화_요금']].head(10)

## 3-2. groupby 유용한 옵션과 팁

In [ ]:
# 팁 1: sort_values()와 함께 사용 - 결과를 정렬하여 보기
# 읍면동별 평균 사용요금 상위 5개
print("=== 평균 사용요금 상위 5개 읍면동 ===")
df.groupby('읍면동명')['사용요금'].mean().sort_values(ascending=False).head(5).round(0)

In [ ]:
# 팁 2: sort=False 옵션 - 원래 등장 순서 유지
# 기본적으로 groupby는 키를 정렬하지만, sort=False로 원래 순서 유지 가능
print("=== sort=True (기본값, 알파벳순) ===")
print(df.groupby('업종')['사용량'].sum().index.tolist())
print()
print("=== sort=False (데이터 등장 순서) ===")
print(df.groupby('업종', sort=False)['사용량'].sum().index.tolist())

In [ ]:
# 팁 3: 구간(bin)으로 그룹화 - pd.cut()과 함께 사용
# 사용량을 구간별로 나누어 분석 (사용량 등급 부여)
df['사용량_등급'] = pd.cut(
    df['사용량'],
    bins=[0, 5, 20, 50, 100, float('inf')],        # 구간 경계값
    labels=['극소량', '소량', '중간', '대량', '초대량'],  # 구간 이름
    right=True,                                       # 오른쪽 포함 (예: 0 < x <= 5)
    include_lowest=True                               # 최소값 포함
)

# 사용량 등급별 통계
df.groupby('사용량_등급', observed=True).agg(
    건수=('사용량', 'count'),
    평균요금=('사용요금', 'mean'),
    총요금=('사용요금', 'sum')
).round(0)

## 3-3. 초급 추가 예제 - groupby 기본기 다지기

In [ ]:
# 초급 1: first()와 last() - 그룹별 첫 번째/마지막 행 확인
# 각 업종에서 데이터 순서상 첫 번째 행과 마지막 행을 추출
print("=== 업종별 첫 번째 데이터 ===")
print(df.groupby('업종').first()[['읍면동명', '사용량', '사용요금']])
print()
print("=== 업종별 마지막 데이터 ===")
print(df.groupby('업종').last()[['읍면동명', '사용량', '사용요금']])

In [ ]:
# 초급 2: nth() - 그룹별 N번째 행 추출
# 각 읍면동의 2번째(인덱스 1) 데이터를 추출
df.groupby('읍면동명').nth(1)[['업종', '사용량', '사용요금']]

In [ ]:
# 초급 3: head()와 tail() - 그룹별 상위/하위 N건
# 각 업종에서 상위 2건씩 추출 (총 업종수 × 2건)
print("=== 업종별 상위 2건씩 ===")
df.groupby('업종').head(2)[['읍면동명', '업종', '사용량', '사용요금']]

In [ ]:
# 초급 4: nunique() - 그룹별 고유값 개수
# 각 읍면동에 몇 가지 업종이 존재하는지 확인
print("=== 읍면동별 업종 종류 수 ===")
df.groupby('읍면동명')['업종'].nunique().sort_values(ascending=False)

In [ ]:
# 초급 5: value_counts()와 groupby 조합
# 각 읍면동별로 업종의 분포(건수)를 확인
# 중구동의 업종별 분포
print("=== 중구동 업종별 건수 ===")
print(df[df['읍면동명'] == '중구동']['업종'].value_counts())
print()
# groupby + value_counts로 모든 읍면동의 업종 분포를 한 번에 확인
print("=== 전체 읍면동별 업종 분포 (상위 15건) ===")
df.groupby('읍면동명')['업종'].value_counts().head(15)

In [ ]:
# 초급 6: idxmin(), idxmax() - 그룹별 최대/최소값의 인덱스 찾기
# 각 업종에서 사용요금이 가장 높은 행의 인덱스 번호를 찾은 후 원본에서 조회
max_idx = df.groupby('업종')['사용요금'].idxmax()
print("=== 각 업종별 최고 요금 수용가의 인덱스 ===")
print(max_idx)
print()

# 해당 인덱스의 원본 행 데이터 확인
print("=== 해당 행의 상세 정보 ===")
df.loc[max_idx][['읍면동명', '업종', '사용량', '사용요금']]

## 3-4. 중급 추가 예제 - 실무에서 자주 쓰는 패턴

In [ ]:
# 중급 1: 조건부 집계 - 특정 조건을 만족하는 건수 세기
# 각 읍면동에서 사용량이 0인 수용가(무사용자)의 비율 계산
df.groupby('읍면동명').agg(
    총건수=('사용량', 'count'),
    무사용건수=('사용량', lambda x: (x == 0).sum()),          # 사용량 == 0인 건수
    무사용비율=('사용량', lambda x: (x == 0).mean() * 100),   # 비율(%)
    대량사용건수=('사용량', lambda x: (x >= 100).sum())        # 사용량 >= 100인 건수
).round(1).sort_values('무사용비율', ascending=False)

In [ ]:
# 중급 2: transform + 불리언 인덱싱 조합
# 각 업종 내에서 평균 이상의 사용요금을 내는 수용가만 필터링
# (행 단위가 아닌, 그룹 평균 기준으로 판단)
df['평균이상여부'] = df['사용요금'] >= df.groupby('업종')['사용요금'].transform('mean')

# 업종별로 평균 이상 비율 확인
print("=== 업종별 평균 이상 요금 비율 ===")
df.groupby('업종')['평균이상여부'].mean().round(3) * 100

In [ ]:
# 중급 3: cumsum(), cumcount() - 그룹 내 누적합, 누적 카운트
# 읍면동별로 사용요금의 누적합과 누적 건수를 계산
# 데이터 순서대로 누적되므로, 진행 경과를 추적할 때 유용
df['읍면동_누적요금'] = df.groupby('읍면동명')['사용요금'].cumsum()
df['읍면동_누적건수'] = df.groupby('읍면동명').cumcount() + 1  # 0부터 시작하므로 +1

# 중구동의 누적 현황 확인 (상위 10건)
df[df['읍면동명'] == '중구동'][['읍면동명', '업종', '사용요금', '읍면동_누적요금', '읍면동_누적건수']].head(10)

In [ ]:
# 중급 4: pct_change() - 그룹 내 변화율 계산
# 같은 읍면동 내에서 이전 행 대비 사용요금 변화율 (%)
df['요금_변화율'] = df.groupby('읍면동명')['사용요금'].pct_change() * 100

# 중구동의 변화율 확인 (NaN이 아닌 행만)
df[df['읍면동명'] == '중구동'][['읍면동명', '업종', '사용요금', '요금_변화율']].dropna().head(10).round(1)

In [ ]:
# 중급 5: diff() - 그룹 내 차분 (이전 값과의 차이)
# 같은 읍면동 내에서 이전 수용가와의 사용량 차이
df['사용량_차분'] = df.groupby('읍면동명')['사용량'].diff()

# 풍산읍의 사용량 차분 확인
df[df['읍면동명'] == '풍산읍'][['읍면동명', '업종', '사용량', '사용량_차분']].head(10)

In [ ]:
# 중급 6: shift() - 그룹 내 값 이동
# 같은 업종 내에서 이전 행의 사용요금을 가져옴 (시차 비교 등에 활용)
df['이전_사용요금'] = df.groupby('업종')['사용요금'].shift(1)

# 현재 요금과 이전 요금 비교
df[df['업종'] == '산업용'][['읍면동명', '업종', '사용요금', '이전_사용요금']].head(10)

In [ ]:
# 중급 7: filter + 다중 조건 조합
# 수용가 수가 1,000건 이상이면서, 동시에 평균 사용량이 30 이상인 읍면동만 추출
df_filtered = df.groupby('읍면동명').filter(
    lambda x: (len(x) >= 1000) & (x['사용량'].mean() >= 30)
)

print(f"원본: {len(df):,}건 → 필터 후: {len(df_filtered):,}건")
print(f"포함된 읍면동: {sorted(df_filtered['읍면동명'].unique())}")
print()
# 필터된 읍면동의 통계 확인
df_filtered.groupby('읍면동명').agg(
    건수=('사용량', 'count'),
    평균사용량=('사용량', 'mean')
).round(1)

In [ ]:
# 중급 8: pd.qcut()으로 동일 개수 구간 분할 후 groupby
# pd.cut()은 동일 간격, pd.qcut()은 동일 개수(분위수) 기반 분할
df['사용요금_분위'] = pd.qcut(
    df['사용요금'],
    q=4,                                           # 4분위 (각 25%씩)
    labels=['하위25%', '중하위25%', '중상위25%', '상위25%'],
    duplicates='drop'                              # 중복 경계값 자동 처리
)

# 분위별 사용량/요금 통계
df.groupby('사용요금_분위', observed=True).agg(
    건수=('사용요금', 'count'),
    평균사용량=('사용량', 'mean'),
    평균요금=('사용요금', 'mean'),
    최대요금=('사용요금', 'max')
).round(0)

## 3-5. 고급 추가 예제 - 심화 분석 패턴

In [ ]:
# 고급 1: 그룹별 이상치(outlier) 탐지 - IQR 방식
# 각 업종 그룹 내에서 IQR 기반 이상치를 찾는 함수
def detect_outliers_iqr(group):
    """IQR(사분위 범위)을 이용한 이상치 탐지"""
    Q1 = group['사용요금'].quantile(0.25)  # 1사분위수
    Q3 = group['사용요금'].quantile(0.75)  # 3사분위수
    IQR = Q3 - Q1                          # 사분위 범위
    upper_bound = Q3 + 1.5 * IQR           # 상한 경계
    # 상한을 초과하는 행만 이상치로 판별
    return group[group['사용요금'] > upper_bound]

# 업종별 이상치 수용가 추출
outliers = df.groupby('업종').apply(detect_outliers_iqr, include_groups=False)

# 업종별 이상치 건수
print("=== 업종별 이상치 건수 ===")
print(outliers.groupby(level=0).size())
print()
print(f"전체 이상치: {len(outliers):,}건 / 전체 {len(df):,}건 ({len(outliers)/len(df)*100:.1f}%)")

In [ ]:
# 고급 2: transform으로 그룹 내 Z-score(표준점수) 계산
# Z-score = (값 - 그룹평균) / 그룹표준편차
# 각 수용가의 요금이 해당 업종 내에서 얼마나 극단적인지 표준화하여 비교
df['요금_zscore'] = df.groupby('업종')['사용요금'].transform(
    lambda x: (x - x.mean()) / x.std()
)

# Z-score가 3 이상인 극단적 수용가 확인 (평균에서 표준편차 3배 이상 벗어남)
extreme = df[df['요금_zscore'] >= 3]
print(f"Z-score >= 3인 극단적 수용가: {len(extreme):,}건")
print()
# 업종별 극단적 수용가 수
extreme.groupby('업종').agg(
    극단건수=('요금_zscore', 'count'),
    평균Z점수=('요금_zscore', 'mean'),
    최대요금=('사용요금', 'max')
).round(2)

In [ ]:
# 고급 3: apply로 그룹별 상관계수 계산
# 각 읍면동에서 사용량과 사용요금의 상관관계가 어떻게 다른지 분석
def calc_correlation(group):
    """그룹 내 사용량-사용요금 상관계수를 계산하는 함수"""
    return pd.Series({
        '상관계수': group['사용량'].corr(group['사용요금']),
        '건수': len(group)
    })

# 읍면동별 사용량-사용요금 상관계수
corr_by_dong = df.groupby('읍면동명').apply(calc_correlation, include_groups=False).round(4)
corr_by_dong.sort_values('상관계수', ascending=False)

In [ ]:
# 고급 4: 그룹별 백분위수 구간 분석
# 업종별로 사용요금의 다양한 백분위수를 한 번에 계산
def percentile_analysis(group):
    """그룹의 주요 백분위수를 Series로 반환하는 함수"""
    return pd.Series({
        '10%ile': group['사용요금'].quantile(0.10),
        '25%ile': group['사용요금'].quantile(0.25),
        '50%ile(중앙값)': group['사용요금'].quantile(0.50),
        '75%ile': group['사용요금'].quantile(0.75),
        '90%ile': group['사용요금'].quantile(0.90),
        '95%ile': group['사용요금'].quantile(0.95),
        '99%ile': group['사용요금'].quantile(0.99)
    })

# 업종별 백분위수 분석
df.groupby('업종').apply(percentile_analysis, include_groups=False).round(0)

In [ ]:
# 고급 5: 다중 groupby 결과 병합 - merge를 활용한 분석
# 읍면동별 전체 통계와 업종별 통계를 각각 계산 후 합쳐서 비교

# 1) 읍면동별 전체 평균 요금
dong_avg = df.groupby('읍면동명')['사용요금'].mean().reset_index()
dong_avg.columns = ['읍면동명', '읍면동_전체평균']

# 2) 읍면동+업종별 평균 요금
dong_type_avg = df.groupby(['읍면동명', '업종'], as_index=False)['사용요금'].mean()
dong_type_avg.columns = ['읍면동명', '업종', '업종별평균']

# 3) 병합하여 비교
merged = pd.merge(dong_type_avg, dong_avg, on='읍면동명')
merged['전체평균대비_차이'] = merged['업종별평균'] - merged['읍면동_전체평균']
merged['전체평균대비_비율'] = (merged['업종별평균'] / merged['읍면동_전체평균'] * 100)

# 전체 평균 대비 업종별 평균이 가장 높은 조합
merged.nlargest(10, '전체평균대비_비율').round(1)

In [ ]:
# 고급 6: 그룹별 Top N 추출의 다양한 방법 비교
# 방법 A: apply + nlargest (유연하지만 느림)
top3_apply = df.groupby('읍면동명').apply(
    lambda x: x.nlargest(3, '사용요금'), include_groups=False
)[['업종', '사용량', '사용요금']]

print("=== 방법 A: apply + nlargest (상위 3개 읍면동만 표시) ===")
print(top3_apply.head(9))
print()

# 방법 B: sort + groupby + head (더 빠른 방법)
top3_sort = df.sort_values('사용요금', ascending=False).groupby('읍면동명').head(3)
print(f"=== 방법 B: sort + head (결과 건수: {len(top3_sort)}) ===")
top3_sort[top3_sort['읍면동명'] == '중구동'][['읍면동명', '업종', '사용량', '사용요금']]

In [ ]:
# 고급 7: 그룹별 사용자 정의 구간 분류 + 교차 집계
# 각 읍면동 그룹 내에서 사용량을 기준으로 등급을 매기는 함수
def assign_grade_within_group(group):
    """그룹 내 사용량 기준으로 A/B/C/D 등급을 부여하는 함수"""
    # 해당 그룹의 분위수 기준으로 등급 부여 (그룹마다 기준이 다름)
    group['사용등급'] = pd.qcut(
        group['사용량'].rank(method='first'),  # rank로 동일값 처리
        q=4,
        labels=['D등급(하위)', 'C등급', 'B등급', 'A등급(상위)']
    )
    return group

# 읍면동별로 각자의 기준으로 등급 부여
df_graded = df.groupby('읍면동명').apply(assign_grade_within_group, include_groups=False)

# 등급별 통계 확인
df_graded.groupby('사용등급', observed=True).agg(
    건수=('사용량', 'count'),
    평균사용량=('사용량', 'mean'),
    평균요금=('사용요금', 'mean')
).round(0)

In [ ]:
# 고급 8: 그룹별 가중 평균 계산
# 단순 평균이 아닌, 사용량을 가중치로 한 가중 평균 요금 계산
# 가중평균 = sum(사용요금 * 사용량) / sum(사용량)
def weighted_avg_fee(group):
    """사용량을 가중치로 한 가중 평균 요금을 계산하는 함수"""
    total_usage = group['사용량'].sum()
    if total_usage == 0:
        return 0
    return (group['사용요금'] * group['사용량']).sum() / total_usage

# 읍면동별 단순평균 vs 가중평균 비교
comparison = pd.DataFrame({
    '단순평균': df.groupby('읍면동명')['사용요금'].mean(),
    '가중평균': df.groupby('읍면동명').apply(weighted_avg_fee, include_groups=False)
}).round(0)

# 차이가 큰 순서대로 정렬 (가중평균과 단순평균의 차이)
comparison['차이'] = comparison['가중평균'] - comparison['단순평균']
comparison.sort_values('차이', ascending=False)

In [ ]:
# 고급 9: 다중 transform을 활용한 파생변수 한 번에 생성
# 여러 그룹 기준에서 동시에 transform을 적용하여 다양한 파생변수 생성
df['업종별_총사용량'] = df.groupby('업종')['사용량'].transform('sum')
df['읍면동별_총사용량'] = df.groupby('읍면동명')['사용량'].transform('sum')
df['업종내_사용량비율'] = (df['사용량'] / df['업종별_총사용량'] * 100)
df['읍면동내_사용량비율'] = (df['사용량'] / df['읍면동별_총사용량'] * 100)

# 결과 확인 - 산업용에서 사용량 비율이 높은 수용가
df[df['업종'] == '산업용'].nlargest(5, '업종내_사용량비율')[
    ['읍면동명', '업종', '사용량', '업종별_총사용량', '업종내_사용량비율']
].round(4)

In [ ]:
# 고급 10: pipe() - groupby 결과에 연쇄적 함수 적용
# pipe()를 사용하면 groupby 결과를 함수 체이닝으로 깔끔하게 처리
def top_n_by_column(grouped_df, column, n=5):
    """그룹별 집계 후 특정 컬럼 기준 상위 N개를 반환하는 함수"""
    return grouped_df.mean(numeric_only=True).nlargest(n, column)

# pipe로 체이닝: groupby → 평균 계산 → 상위 5개 추출을 한 줄로
print("=== 평균 사용요금 상위 5개 읍면동 (pipe 활용) ===")
df.groupby('읍면동명').pipe(top_n_by_column, column='사용요금', n=5)[['사용량', '사용요금']].round(0)

## 3-6. 실전 연습문제

> 아래 연습문제를 직접 풀어보세요. 초급/중급/고급으로 난이도가 구분되어 있습니다.

### [초급] 연습문제 1
**읍면동별로 "가정용" 수용가의 평균 사용요금을 구하고, 상위 5개 지역을 출력하세요.**

> 힌트: 먼저 업종이 '가정용'인 데이터를 필터링한 후 groupby를 적용합니다.

In [ ]:
# 연습문제 1 풀이 공간
# 아래에 코드를 작성하세요




<details>
<summary>💡 연습문제 1 정답 (클릭하여 확인)</summary>

```python
df[df['업종'] == '가정용'].groupby('읍면동명')['사용요금'].mean().sort_values(ascending=False).head(5).round(0)
```
</details>

### [초급] 연습문제 2
**읍면동별로 사용량의 합계, 평균, 중앙값, 건수를 Named Aggregation으로 구하세요.**

> 힌트: `agg(이름=('컬럼', '함수'))` 형태를 사용합니다.

In [ ]:
# 연습문제 2 풀이 공간
# 아래에 코드를 작성하세요




<details>
<summary>💡 연습문제 2 정답 (클릭하여 확인)</summary>

```python
df.groupby('읍면동명').agg(
    사용량합계=('사용량', 'sum'),
    사용량평균=('사용량', 'mean'),
    사용량중앙값=('사용량', 'median'),
    건수=('사용량', 'count')
).round(1)
```
</details>

### [중급] 연습문제 3
**transform()을 사용하여 각 수용가의 사용요금이 해당 읍면동 평균의 몇 배인지 계산하는 '요금배율' 컬럼을 추가하세요. 요금배율이 가장 높은 상위 5건을 출력하세요.**

> 힌트: `요금배율 = 사용요금 / 그룹평균`

In [ ]:
# 연습문제 3 풀이 공간
# 아래에 코드를 작성하세요




<details>
<summary>💡 연습문제 3 정답 (클릭하여 확인)</summary>

```python
df['요금배율'] = df['사용요금'] / df.groupby('읍면동명')['사용요금'].transform('mean')
df.nlargest(5, '요금배율')[['읍면동명', '업종', '사용요금', '요금배율']].round(1)
```
</details>

### [중급] 연습문제 4
**각 읍면동에서 업종별 총 사용요금이 가장 높은 업종을 찾으세요.**
> 즉, 22개 읍면동 각각에서 "어떤 업종이 가장 많은 요금을 납부하는지" 확인합니다.

> 힌트: `idxmax()`를 활용하세요.

In [ ]:
# 연습문제 5 풀이 공간
# 아래에 코드를 작성하세요




<details>
<summary>💡 연습문제 5 정답 (클릭하여 확인)</summary>

```python
# 읍면동-업종별 총 요금을 피벗테이블로 만든 뒤 각 행에서 최대 컬럼을 찾기
pivot_fee = df.groupby(['읍면동명', '업종'])['사용요금'].sum().unstack(fill_value=0)
result = pivot_fee.idxmax(axis=1)
print("=== 읍면동별 최대 요금 업종 ===")
print(result)
```
</details>

### [중급] 연습문제 5
**filter()를 사용하여 "평균 사용량이 전체 평균 사용량보다 높은 읍면동"만 추출하세요. 추출된 읍면동의 이름과 평균 사용량을 출력하세요.**

> 힌트: 전체 평균은 `df['사용량'].mean()`으로 먼저 구합니다.

In [ ]:
# 연습문제 5 풀이 공간
# 아래에 코드를 작성하세요




<details>
<summary>💡 연습문제 5 정답 (클릭하여 확인)</summary>

```python
overall_avg = df['사용량'].mean()
print(f"전체 평균 사용량: {overall_avg:.1f}")

df_above_avg = df.groupby('읍면동명').filter(lambda x: x['사용량'].mean() > overall_avg)
print(f"포함된 읍면동: {sorted(df_above_avg['읍면동명'].unique())}")

df_above_avg.groupby('읍면동명')['사용량'].mean().sort_values(ascending=False).round(1)
```
</details>

### [고급] 연습문제 6
**apply()를 사용하여 각 읍면동에서 "사용요금 상위 10%에 해당하는 수용가"의 평균 사용량과 평균 요금을 구하세요.**

> 힌트: 그룹 내에서 `quantile(0.9)`를 구한 뒤, 그 이상인 행만 필터링합니다.

In [ ]:
# 연습문제 6 풀이 공간
# 아래에 코드를 작성하세요




<details>
<summary>💡 연습문제 6 정답 (클릭하여 확인)</summary>

```python
def top10_pct_stats(group):
    threshold = group['사용요금'].quantile(0.9)
    top10 = group[group['사용요금'] >= threshold]
    return pd.Series({
        '상위10%_건수': len(top10),
        '상위10%_평균사용량': top10['사용량'].mean(),
        '상위10%_평균요금': top10['사용요금'].mean()
    })

df.groupby('읍면동명').apply(top10_pct_stats, include_groups=False).round(0)
```
</details>

### [고급] 연습문제 7
**각 업종별로 사용요금의 Z-score가 2 이상인 수용가(고액 사용자)의 비율(%)을 구하세요. 어떤 업종에 고액 사용자 비율이 가장 높은가요?**

> 힌트: transform으로 Z-score를 계산한 뒤, 조건부 집계(lambda)를 활용합니다.

In [ ]:
# 연습문제 7 풀이 공간
# 아래에 코드를 작성하세요




<details>
<summary>💡 연습문제 7 정답 (클릭하여 확인)</summary>

```python
df['zscore'] = df.groupby('업종')['사용요금'].transform(lambda x: (x - x.mean()) / x.std())

df.groupby('업종').agg(
    총건수=('zscore', 'count'),
    고액사용자수=('zscore', lambda x: (x >= 2).sum()),
    고액사용자비율=('zscore', lambda x: (x >= 2).mean() * 100)
).round(2).sort_values('고액사용자비율', ascending=False)
```
</details>

---
## 수업 정리

### groupby 핵심 메서드 요약

| 메서드 | 반환 크기 | 용도 | 대표 예시 |
|--------|----------|------|----------|
| `sum/mean/count/...()` | 그룹 수 | 단일 집계 | 업종별 평균 요금 |
| `agg()` | 그룹 수 | 다중 집계 | 합계+평균+최대 동시 계산 |
| `transform()` | 원본과 동일 | 브로드캐스트 | 그룹 평균 컬럼 추가, Z-score |
| `filter()` | 원본 이하 | 그룹 필터링 | 건수 1만 이상 지역만 |
| `apply()` | 자유 | 범용 함수 적용 | 그룹별 Top 3, 상관계수 |

### 유용한 그룹 내 연산

| 메서드 | 용도 |
|--------|------|
| `cumsum()` / `cumcount()` | 그룹 내 누적합 / 누적 카운트 |
| `shift()` / `diff()` | 그룹 내 값 이동 / 차분 |
| `pct_change()` | 그룹 내 변화율 |
| `rank()` | 그룹 내 순위 |
| `nlargest()` / `nsmallest()` | 그룹별 Top/Bottom N |

### 자주 쓰이는 옵션
- `as_index=False`: 그룹 키를 인덱스가 아닌 컬럼으로 유지
- `sort=False`: 그룹 키 정렬하지 않음 (성능 향상)
- `observed=True`: Categorical 컬럼에서 실제 존재하는 값만 사용
- `include_groups=False`: apply/transform에서 그룹 키 컬럼 제외

수고하셨습니다! 🎉